In [ ]:
setwd("./real_data_CSA_AFR")

In [2]:
library(MRAPSS)
library(data.table)
library(MASS)
library(ggpubr)

basdir = getwd()

Loading required package: ggplot2



### format CSA/EUR data

In [3]:
info_csa = read.csv('CSA_data_info.csv', row.names = 1)

In [4]:
head(info_csa)

,Phenotype,Category,code,wget,n_cases_CSA,n_cases_EAS,n_cases_EUR,n_controls_CSA,n_controls_EAS,n_controls_EUR,n_CSA,n_EUR,address
,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<int>,<chr>
2,Asthma,Immune,J45,wget https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/icd10-J45-both_sexes.tsv.bgz,900,149,31030,7976,2560,389501,8876,420531,icd10-J45-both_sexes.tsv
3,Chronic ischaemic heart disease,Cardiometabolic,I25,wget https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/icd10-I25-both_sexes.tsv.bgz,1205,83,31148,7671,2626,389383,8876,420531,icd10-I25-both_sexes.tsv
4,T2D,Cardiometabolic,250.2,wget https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/phecode-250.2-both_sexes.tsv.bgz,1664,152,22768,7181,2555,396181,8845,418949,phecode-250.2-both_sexes.tsv
5,IGF-1,Anthropometric,30770,wget https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/biomarkers-30770-both_sexes-irnt.tsv.bgz,8364,2560,398797,NA,NA,NA,8364,398797,biomarkers-30770-both_sexes-irnt.tsv
6,SHBG,Anthropometric,30830,wget https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/biomarkers-30830-both_sexes-irnt.tsv.bgz,7643,2327,363650,NA,NA,NA,7643,363650,biomarkers-30830-both_sexes-irnt.tsv
7,Vitamin D,Anthropometric,30890,wget https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/biomarkers-30890-both_sexes-irnt.tsv.bgz,7380,2484,383324,NA,NA,NA,7380,383324,biomarkers-30890-both_sexes-irnt.tsv


In [5]:
variants = data.table::fread('./full_variant_qc_metrics.txt')

In [ ]:
for(i in 1:dim(info_csa)[1]){
    
    phecode =  info_csa[['code']][i]
    
    print(paste0("Trait: ", phecode, "..."))
    
    data = data.table::fread(paste0('./raw_data/', info_csa[['address']][i]))
    
    if(sum(!(data$chr == variants$chrom & data$pos == variants$pos)) != 0){
        print('Data and variants not fully matched. Skip...')
        next
    }
    
    data$rsid = variants$rsid
    
    if(is.na(info_csa[['n_controls_CSA']][i])){
        
        data = data[, c('chr','pos','rsid','ref','alt','af_CSA','af_EUR','beta_CSA','beta_EUR','se_CSA','se_EUR','neglog10_pval_CSA','neglog10_pval_EUR','low_confidence_CSA','low_confidence_EUR')]
        data_sub = data[!(is.na(data$low_confidence_CSA) | is.na(data$low_confidence_EUR)),]

        data_sub = data_sub[(data_sub$low_confidence_CSA == FALSE) & (data_sub$low_confidence_EUR == FALSE),]

        data_sub$pval_CSA = 10^(-data_sub$neglog10_pval_CSA)
        data_sub$pval_EUR = 10^(-data_sub$neglog10_pval_EUR)

        data_sub = as.data.frame(data_sub)

        dat_EUR = format_data(data_sub,
                              snp_col = "rsid",
                              b_col = "beta_EUR",
                              se_col = "se_EUR",
                              A1_col = "alt",
                              A2_col = "ref",
                              p_col = "pval_EUR",
                              freq_col = "af_EUR",
                              n = info_csa[['n_EUR']][i],
                              min_freq = 0.01)

        dat_CSA = format_data(data_sub,
                              snp_col = "rsid",
                              b_col = "beta_CSA",
                              se_col = "se_CSA",
                              A1_col = "alt",
                              A2_col = "ref",
                              p_col = "pval_CSA",
                              freq_col = "af_CSA",
                              n = info_csa[['n_CSA']][i],
                              min_freq = 0.01)

    }else{
        
        data = data[, c('chr','pos','rsid','ref','alt','af_cases_CSA','af_cases_EUR','beta_CSA','beta_EUR','se_CSA','se_EUR','neglog10_pval_CSA','neglog10_pval_EUR','low_confidence_CSA','low_confidence_EUR')]
        data_sub = data[!(is.na(data$low_confidence_CSA) | is.na(data$low_confidence_EUR)),]

        data_sub = data_sub[(data_sub$low_confidence_CSA == FALSE) & (data_sub$low_confidence_EUR == FALSE),]

        data_sub$pval_CSA = 10^(-data_sub$neglog10_pval_CSA)
        data_sub$pval_EUR = 10^(-data_sub$neglog10_pval_EUR)

        data_sub = as.data.frame(data_sub)

        dat_EUR = format_data(data_sub,
                              snp_col = "rsid",
                              b_col = "beta_EUR",
                              se_col = "se_EUR",
                              A1_col = "alt",
                              A2_col = "ref",
                              p_col = "pval_EUR",
                              freq_col = "af_cases_EUR",
                              n = info_csa[['n_EUR']][i],
                              min_freq = 0.01)

        dat_CSA = format_data(data_sub,
                              snp_col = "rsid",
                              b_col = "beta_CSA",
                              se_col = "se_CSA",
                              A1_col = "alt",
                              A2_col = "ref",
                              p_col = "pval_CSA",
                              freq_col = "af_cases_CSA",
                              n = info_csa[['n_CSA']][i],
                              min_freq = 0.01)

    }
                  
    write.table(dat_EUR, file=paste0("./formatted_data/EUR_",phecode), quote = F, col.names = T, row.names = F)
    write.table(dat_CSA, file=paste0("./formatted_data/CSA_",phecode), quote = F, col.names = T, row.names = F)

}

### format AFR data

In [27]:
info_afr = read.csv('AFR_data_info.csv', row.names = 1)

In [28]:
info_afr$address =sub("\\.bgz$", "", basename(info_afr$AFR.data.link))
info_afr[dim(info_afr)[1], 'address'] = 'AFR_alb_new.txt'

In [30]:
info_afr

,EUR.phenotype,EUR.phecode,EUR.samples,EUR.cases,EUR.controls,EUR.data.link,AFR.phenotype,AFR.phecode,AFR.samples,AFR.cases,AFR.controls,AFR.data.link,address
,<chr>,<chr>,<int>,<int>,<int>,<chr>,<chr>,<chr>,<int>,<int>,<int>,<chr>,<chr>
8,Asthma,J45,420531,31030,389501,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/icd10-J45-both_sexes.tsv.bgz,Asthma,J45,6636,573,6063,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/icd10-J45-both_sexes.tsv.bgz,icd10-J45-both_sexes.tsv
9,Chronic ischaemic heart disease,I25,420531,31148,389383,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/icd10-I25-both_sexes.tsv.bgz,Chronic ischaemic heart disease,I25,6636,302,6334,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/icd10-I25-both_sexes.tsv.bgz,icd10-I25-both_sexes.tsv
10,T2D,250.2,418949,22768,396181,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/phecode-250.2-both_sexes.tsv.bgz,T2D,250.2,6589,796,5793,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/phecode-250.2-both_sexes.tsv.bgz,phecode-250.2-both_sexes.tsv
11,IGF-1,30770,398797,398797,NA,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/biomarkers-30770-both_sexes-irnt.tsv.bgz,IGF-1,30770,6178,6178,NA,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/biomarkers-30770-both_sexes-irnt.tsv.bgz,biomarkers-30770-both_sexes-irnt.tsv
12,SHBG,30830,363650,363650,NA,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/biomarkers-30830-both_sexes-irnt.tsv.bgz,SHBG,30830,5709,5709,NA,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/biomarkers-30830-both_sexes-irnt.tsv.bgz,biomarkers-30830-both_sexes-irnt.tsv
13,Vitamin D,30890,383324,383324,NA,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/biomarkers-30890-both_sexes-irnt.tsv.bgz,Vitamin D,30890,6046,6046,NA,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/biomarkers-30890-both_sexes-irnt.tsv.bgz,biomarkers-30890-both_sexes-irnt.tsv
15,C-reactive protein,30710,400094,400094,NA,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/biomarkers-30710-both_sexes-irnt.tsv.bgz,C-reactive protein,30710,6203,6203,NA,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/biomarkers-30710-both_sexes-irnt.tsv.bgz,biomarkers-30710-both_sexes-irnt.tsv
18,LDL direct,30780,400223,400223,NA,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/biomarkers-30780-both_sexes-irnt.tsv.bgz,LDL direct,30780,6200,6200,NA,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/biomarkers-30780-both_sexes-irnt.tsv.bgz,biomarkers-30780-both_sexes-irnt.tsv
19,Triglycerides,30870,400639,400639,NA,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/biomarkers-30870-both_sexes-irnt.tsv.bgz,Triglycerides,30870,6211,6211,NA,https://pan-ukb-us-east-1.s3.amazonaws.com/sumstats_flat_files/biomarkers-30870-both_sexes-irnt.tsv.bgz,biomarkers-30870-both_sexes-irnt.tsv


In [31]:
for(i in 1:(dim(info_afr)[1]-1)){
    
    phecode =  info_afr[['EUR.phecode']][i]
    
    print(paste0("Trait: ", phecode, "..."))
    
    data = data.table::fread(paste0('./raw_data/', info_afr[['address']][i]))
    
    if(sum(!(data$chr == variants$chrom & data$pos == variants$pos)) != 0){
        print('Data and variants not fully matched. Skip...')
        next
    }
    
    data$rsid = variants$rsid
    
    if(is.na(info_afr[['AFR.controls']][i])){
        
        data = data[, c('chr','pos','rsid','ref','alt','af_AFR','af_EUR','beta_AFR','beta_EUR','se_AFR','se_EUR','neglog10_pval_AFR','neglog10_pval_EUR','low_confidence_AFR','low_confidence_EUR')]
        data_sub = data[!(is.na(data$low_confidence_AFR) | is.na(data$low_confidence_EUR)),]

        data_sub = data_sub[(data_sub$low_confidence_AFR == FALSE) & (data_sub$low_confidence_EUR == FALSE),]

        data_sub$pval_AFR = 10^(-data_sub$neglog10_pval_AFR)
        data_sub$pval_EUR = 10^(-data_sub$neglog10_pval_EUR)

        data_sub = as.data.frame(data_sub)

        dat_AFR = format_data(data_sub,
                              snp_col = "rsid",
                              b_col = "beta_AFR",
                              se_col = "se_AFR",
                              A1_col = "alt",
                              A2_col = "ref",
                              p_col = "pval_AFR",
                              freq_col = "af_AFR",
                              n = info_afr[['AFR.samples']][i],
                              min_freq = 0.01)

    }else{
        
        data = data[, c('chr','pos','rsid','ref','alt','af_cases_AFR','af_cases_EUR','beta_AFR','beta_EUR','se_AFR','se_EUR','neglog10_pval_AFR','neglog10_pval_EUR','low_confidence_AFR','low_confidence_EUR')]
        data_sub = data[!(is.na(data$low_confidence_AFR) | is.na(data$low_confidence_EUR)),]

        data_sub = data_sub[(data_sub$low_confidence_AFR == FALSE) & (data_sub$low_confidence_EUR == FALSE),]

        data_sub$pval_AFR = 10^(-data_sub$neglog10_pval_AFR)

        data_sub = as.data.frame(data_sub)
        
        dat_AFR = format_data(data_sub,
                              snp_col = "rsid",
                              b_col = "beta_AFR",
                              se_col = "se_AFR",
                              A1_col = "alt",
                              A2_col = "ref",
                              p_col = "pval_AFR",
                              freq_col = "af_cases_AFR",
                              n = info_afr[['AFR.samples']][i],
                              min_freq = 0.01)
              
    }
    
    
    write.table(dat_AFR, file=paste0("./formatted_data/AFR_",phecode), quote = F, col.names = T, row.names = F)

    
}

[1] "Trait: J45..."


Begin formatting .... 

The raw data set has 13687571 dat lines

Remove ambiguous SNPs ..., remaining 11782937 SNPs.

Remove SNPs in MHC region ..., remaining 11736229 SNPs.

Remove duplicated SNPs ..., remaining 11716738 SNPs.

Merge SNPs with the hapmap3 snplist ..., remaining 1201683 SNPs.

Remove SNPs with alleles not matched with the hapmap3 snplist, remaining 1201672 SNPs.

Remove SNPs with MAF less than0.01, remaining 1170540 SNPs.

Generating sample size from specified sample size

Remove SNPs with p-value < 0 or p-value > 1, remaining 1170540 SNPs.

Inferring z score from p value and b ...

Remove missing values, remaining 1170540 SNPs.

Remove SNPs with sample size 5 standard deviations away from the mean, remaining 1170540 SNPs.

Remove SNPs with chi2 > chi2_max ... , remaining 1170540 SNPs.

The formatted data has 1170540 dat lines. 




[1] "Trait: I25..."


Begin formatting .... 

The raw data set has 12857472 dat lines

Remove ambiguous SNPs ..., remaining 11067040 SNPs.

Remove SNPs in MHC region ..., remaining 11022496 SNPs.

Remove duplicated SNPs ..., remaining 11004419 SNPs.

Merge SNPs with the hapmap3 snplist ..., remaining 1188666 SNPs.

Remove SNPs with alleles not matched with the hapmap3 snplist, remaining 1188655 SNPs.

Remove SNPs with MAF less than0.01, remaining 1165482 SNPs.

Generating sample size from specified sample size

Remove SNPs with p-value < 0 or p-value > 1, remaining 1165482 SNPs.

Inferring z score from p value and b ...

Remove missing values, remaining 1165482 SNPs.

Remove SNPs with sample size 5 standard deviations away from the mean, remaining 1165482 SNPs.

Remove SNPs with chi2 > chi2_max ... , remaining 1165482 SNPs.

The formatted data has 1165482 dat lines. 




[1] "Trait: 250.2..."


Begin formatting .... 

The raw data set has 13317193 dat lines

Remove ambiguous SNPs ..., remaining 11465691 SNPs.

Remove SNPs in MHC region ..., remaining 11418521 SNPs.

Remove duplicated SNPs ..., remaining 11399458 SNPs.

Merge SNPs with the hapmap3 snplist ..., remaining 1201721 SNPs.

Remove SNPs with alleles not matched with the hapmap3 snplist, remaining 1201710 SNPs.

Remove SNPs with MAF less than0.01, remaining 1167401 SNPs.

Generating sample size from specified sample size

Remove SNPs with p-value < 0 or p-value > 1, remaining 1167401 SNPs.

Inferring z score from p value and b ...

Remove missing values, remaining 1167401 SNPs.

Remove SNPs with sample size 5 standard deviations away from the mean, remaining 1167401 SNPs.

Remove SNPs with chi2 > chi2_max ... , remaining 1167401 SNPs.

The formatted data has 1167401 dat lines. 




... (14 outputs left out)

[1] "Trait: 272..."


Begin formatting .... 

The raw data set has 14349383 dat lines

Remove ambiguous SNPs ..., remaining 12348260 SNPs.

Remove SNPs in MHC region ..., remaining 12300052 SNPs.

Remove duplicated SNPs ..., remaining 12279568 SNPs.

Merge SNPs with the hapmap3 snplist ..., remaining 1202032 SNPs.

Remove SNPs with alleles not matched with the hapmap3 snplist, remaining 1202019 SNPs.

Remove SNPs with MAF less than0.01, remaining 1167388 SNPs.

Generating sample size from specified sample size

Remove SNPs with p-value < 0 or p-value > 1, remaining 1167388 SNPs.

Inferring z score from p value and b ...

Remove missing values, remaining 1167388 SNPs.

Remove SNPs with sample size 5 standard deviations away from the mean, remaining 1167388 SNPs.

Remove SNPs with chi2 > chi2_max ... , remaining 1167388 SNPs.

The formatted data has 1167388 dat lines. 




[1] "Trait: 30140..."


Begin formatting .... 

The raw data set has 15536053 dat lines

Remove ambiguous SNPs ..., remaining 13366365 SNPs.

Remove SNPs in MHC region ..., remaining 13316541 SNPs.

Remove duplicated SNPs ..., remaining 13294555 SNPs.

Merge SNPs with the hapmap3 snplist ..., remaining 1205638 SNPs.

Remove SNPs with alleles not matched with the hapmap3 snplist, remaining 1205628 SNPs.

Remove SNPs with MAF less than0.01, remaining 1167958 SNPs.

Generating sample size from specified sample size

Remove SNPs with p-value < 0 or p-value > 1, remaining 1167958 SNPs.

Inferring z score from p value and b ...

Remove missing values, remaining 1167958 SNPs.

Remove SNPs with sample size 5 standard deviations away from the mean, remaining 1167958 SNPs.

Remove SNPs with chi2 > chi2_max ... , remaining 1167909 SNPs.

The formatted data has 1167909 dat lines. 




#### albumin (AFR)

In [12]:
# Read in dbSNP153 hg19 data
dbSNP <- readr::read_delim("../real_data_EAS/raw_data/EUR/dbSnp153Common.hg19.bed", col_names = FALSE)[, c(1:4)]
colnames(dbSNP) <- c("CHR", "START", "END", "rsID")

setDT(dbSNP)

setkey(dbSNP, CHR, END)

Rows: 14908375 Columns: 17
── Column specification ────────────────────────────────────────────────────────────────────────────────────────────
Delimiter: "\t"
chr (9): X1, X4, X5, X7, X10, X11, X12, X14, X15
dbl (8): X2, X3, X6, X8, X9, X13, X16, X17

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


In [13]:
for(i in dim(info_afr)[1]){
    
    phecode =  info_afr[['EUR.phecode']][i]
    
    print(paste0("Trait: ", phecode, "..."))

    #data = data.table::fread(paste0('./raw_data/', 'AFR_alb_new.txt'))
    data = data.table::fread(paste0('./raw_data/', info_afr[['address']][i]))
    
    all_cols = colnames(data)

    data = data[!is.na(data$beta_re),]

    data[, c("CHR", "POS", "allele1", "allele2") := tstrsplit(snpid, ":", fixed = TRUE)]

    data$POS = as.numeric(data$POS)

    data$af = NA

    n = 0
    if('no_uganda' %in% all_cols){
        n1 = unique(data$no_uganda)
        if(length(n1)==1){
            n = n + n1
        }else{
            print('sample size not unique in uganda...')
            next
        }

        data$af = data$af_uganda
    }

    if('no_AADM' %in% all_cols){
        n2 = unique(data$no_AADM)
        if(length(n2)==1){
            n = n + n2
        }else{
            print('sample size not unique i AADM...')
            next
        }

        data[is.na(data$af),]$af = data[is.na(data$af),]$af_AADM
    }

    if('no_DCC' %in% all_cols){
        n3 = unique(data$no_DCC)
        if(length(n3)==1){
            n = n + n3
        }else{
            print('sample size not unique in DCC...')
            next
        }

        data[is.na(data$af),]$af = data[is.na(data$af),]$af_DCC
    }

    if('no_DDS' %in% all_cols){
        n4 = unique(data$no_DDS)
        if(length(n4)==1){
            n = n + n4
        }else{
            print('sample size not unique in DDS...')
            next
        }

        data[is.na(data$af),]$af = data[is.na(data$af),]$af_DDS

    }


    data$CHR_modified = paste0("chr", data$CHR)
    data$SNPID = data$snpid

    setDT(data)
    data[dbSNP, on = .(CHR_modified = CHR, POS = END), SNPID := i.rsID]

    data[is.na(data$SNPID)]$SNPID = data[is.na(data$SNPID)]$snpid
    data[(data$SNPID=='NA')]$SNPID = data[(data$SNPID=='NA')]$snpid


    data = format_data(data,
                      snp_col = "SNPID",
                      b_col = "beta_re",
                      se_col = "se_re",
                      A1_col = "allele1",
                      A2_col = "allele2",
                      p_col = "pval_re",
                      freq_col = "af",
                      n = n,
                      min_freq = 0.01)

    write.table(data, file=paste0("./formatted_data/AFR_",phecode), quote = F, col.names = T, row.names = F)
    
}

[1] "Trait: 30600..."


Begin formatting .... 

The raw data set has 25401794 dat lines

the other allele column has some values that are not A/C/T/G. Remove these SNPs..., remaining 25401793 SNPs.

Remove ambiguous SNPs ..., remaining 21804986 SNPs.

Remove SNPs in MHC region ..., remaining 21765934 SNPs.

Remove duplicated SNPs ..., remaining 21566595 SNPs.

Merge SNPs with the hapmap3 snplist ..., remaining 1155247 SNPs.

Remove SNPs with alleles not matched with the hapmap3 snplist, remaining 1155158 SNPs.

Remove SNPs with MAF less than0.01, remaining 1135097 SNPs.

Generating sample size from specified sample size

Remove SNPs with p-value < 0 or p-value > 1, remaining 1135097 SNPs.

Inferring z score from p value and b ...

Remove missing values, remaining 1135097 SNPs.

Remove SNPs with sample size 5 standard deviations away from the mean, remaining 1135097 SNPs.

Remove SNPs with chi2 > chi2_max ... , remaining 1135097 SNPs.

The formatted data has 1135097 dat lines. 




#### albumin (EUR)

In [42]:
variants_eas = fread('../real_data_EAS/raw_data/EUR/variants.tsv')

In [43]:
head(variants_eas)

variant,chr,pos,ref,alt,rsid,varid,consequence,consequence_category,info,⋯,p_hwe,n_called,n_not_called,n_hom_ref,n_het,n_hom_var,n_non_ref,r_heterozygosity,r_het_hom_var,r_expected_het_frequency
<chr>,<chr>,<int>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,⋯,<dbl>,<int>,<int>,<int>,<int>,<int>,<int>,<dbl>,<dbl>,<dbl>
1:15791:C:T,1,15791,C,T,rs547522712,1:15791_C_T,splice_region_variant,missense,0.861678,⋯,0.5000000,361194,0,361194,0,0,0,0.00000e+00,NA,0.00000e+00
1:69487:G:A,1,69487,G,A,rs568226429,1:69487_G_A,missense_variant,missense,0.956975,⋯,0.5000040,361194,0,361190,4,0,4,1.10744e-05,NA,1.10743e-05
1:69569:T:C,1,69569,T,C,rs2531267,1:69569_T_C,missense_variant,missense,0.831664,⋯,0.5063150,361194,0,361058,136,0,136,3.76529e-04,NA,3.76459e-04
1:139853:C:T,1,139853,C,T,rs533633326,1:139853_C_T,splice_region_variant,missense,0.985255,⋯,0.5000040,361194,0,361190,4,0,4,1.10744e-05,NA,1.10743e-05
1:692794:CA:C,1,692794,CA,C,1:692794_CA_C,1:692794_CA_C,upstream_gene_variant,non_coding,0.824483,⋯,0.0314373,361188,6,286823,70224,4141,74365,1.94425e-01,16.9582,1.93734e-01
1:693731:A:G,1,693731,A,G,rs12238997,1:693731_A_G,upstream_gene_variant,non_coding,0.875969,⋯,0.1641640,361192,2,281659,74692,4841,79533,2.06793e-01,15.4290,2.06315e-01


In [44]:
head(data_UKB)

variant,minor_allele,minor_AF,low_confidence_variant,n_complete_samples,AC,ytx,beta,se,tstat,pval
<chr>,<chr>,<dbl>,<lgl>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1:15791:C:T,T,6.2194e-09,TRUE,315268,3.9216e-03,1.7110e-01,-606.650000,650.630000,-0.93240,0.351130
1:69487:G:A,A,6.5739e-06,TRUE,315268,4.1451e+00,1.8295e+02,-1.170400,1.274400,-0.91838,0.358420
1:69569:T:C,C,1.9103e-04,TRUE,315268,1.2045e+02,5.4602e+03,0.066636,0.242320,0.27499,0.783320
1:139853:C:T,T,6.4869e-06,TRUE,315268,4.0902e+00,1.8050e+02,-1.160200,1.274500,-0.91036,0.362630
1:692794:CA:C,C,1.1053e-01,FALSE,315268,6.9691e+04,3.1521e+06,-0.012252,0.011300,-1.08430,0.278250
1:693731:A:G,G,1.1584e-01,FALSE,315268,7.3040e+04,3.3031e+06,-0.020217,0.010674,-1.89390,0.058233


In [45]:
phecode = '30600'

print(paste0("Trait: ", phecode, "..."))

UKB_address = "./raw_data/30600.tsv"
data_UKB = fread(UKB_address)

check1 = sum(data_UKB$variant != variants_eas$variant)

if(check1 != 0){
    print('Need more careful alignment with variants.tsv... skip')
}

data_UKB$SNP = variants_eas$rsid

data_UKB$A1 = variants_eas$alt
data_UKB$A2 = variants_eas$ref

dat_UKB = format_data(data_UKB,
              snp_col = "SNP",
              b_col = "beta",
              se_col = "se",
              A1_col = "A1",
              A2_col = "A2",
              p_col = "pval",
              freq_col = "minor_AF",
              n = 315268,
              min_freq = 0.01)

write.table(dat_UKB, file="./formatted_data/EUR_30600", quote = F, col.names = T, row.names = F)

[1] "Trait: 30600..."


Begin formatting .... 

The raw data set has 13791467 dat lines

Remove ambiguous SNPs ..., remaining 11862242 SNPs.

Remove SNPs in MHC region ..., remaining 11815242 SNPs.

Remove duplicated SNPs ..., remaining 11798741 SNPs.

Merge SNPs with the hapmap3 snplist ..., remaining 1197531 SNPs.

Remove SNPs with alleles not matched with the hapmap3 snplist, remaining 1197510 SNPs.

Remove SNPs with MAF less than0.01, remaining 1173750 SNPs.

Generating sample size from specified sample size

Remove SNPs with p-value < 0 or p-value > 1, remaining 1173750 SNPs.

Inferring z score from p value and b ...

Remove missing values, remaining 1173750 SNPs.

Remove SNPs with sample size 5 standard deviations away from the mean, remaining 1173750 SNPs.

Remove SNPs with chi2 > chi2_max ... , remaining 1173710 SNPs.

The formatted data has 1173710 dat lines. 




### Case study: SHBG in EAS

In [46]:
i = which(info_csa$code == '30830')

phecode =  info_csa[['code']][i]

print(paste0("Trait: ", phecode, "..."))

data = data.table::fread(paste0('./raw_data/', info_csa[['address']][i]))

if(sum(!(data$chr == variants$chrom & data$pos == variants$pos)) != 0){
    print('Data and variants not fully matched. Skip...')
    next
}

data$rsid = variants$rsid
data = data[, c('chr','pos','rsid','ref','alt','af_EAS','af_EUR','beta_EAS','beta_EUR','se_EAS','se_EUR','neglog10_pval_EAS','neglog10_pval_EUR','low_confidence_EAS','low_confidence_EUR')]
data_sub = data[!(is.na(data$low_confidence_EAS) | is.na(data$low_confidence_EUR)),]

data_sub = data_sub[(data_sub$low_confidence_EAS == FALSE) & (data_sub$low_confidence_EUR == FALSE),]

data_sub$pval_EAS = 10^(-data_sub$neglog10_pval_EAS)

data_sub = as.data.frame(data_sub)


dat_EAS = format_data(data_sub,
                      snp_col = "rsid",
                      b_col = "beta_EAS",
                      se_col = "se_EAS",
                      A1_col = "alt",
                      A2_col = "ref",
                      p_col = "pval_EAS",
                      freq_col = "af_EAS",
                      n = 2327,
                      min_freq = 0.01)

write.table(dat_EAS, file=paste0("./formatted_data/EAS_",phecode), quote = F, col.names = T, row.names = F)

[1] "Trait: 30830..."


Begin formatting .... 

The raw data set has 9236369 dat lines

Remove ambiguous SNPs ..., remaining 7949990 SNPs.

Remove SNPs in MHC region ..., remaining 7912413 SNPs.

Remove duplicated SNPs ..., remaining 7901459 SNPs.

Merge SNPs with the hapmap3 snplist ..., remaining 1164777 SNPs.

Remove SNPs with alleles not matched with the hapmap3 snplist, remaining 1164755 SNPs.

Remove SNPs with MAF less than0.01, remaining 1117477 SNPs.

Generating sample size from specified sample size

Remove SNPs with p-value < 0 or p-value > 1, remaining 1117477 SNPs.

Inferring z score from p value and b ...

Remove missing values, remaining 1117477 SNPs.

Remove SNPs with sample size 5 standard deviations away from the mean, remaining 1117477 SNPs.

Remove SNPs with chi2 > chi2_max ... , remaining 1117477 SNPs.

The formatted data has 1117477 dat lines. 


